# 21 - ¿La resolución 1024 ayuda, o nunca se probó?

## Por qué existe este notebook

La lectura que circulaba era "probamos 1024 y salió peor". Leyendo los `config.json`
y contando las filas de `train_log.csv`, esos runs cambiaban **cuatro** cosas a la vez:

| run | res | bs | lr | patience | épocas | F1 |
|---|---|---|---|---|---|---|
| `supervised_1024` | 1024 | 1 | 3e-4 | 50 | 89 | .7501 |
| `1024ANTERIORPATIENCE20` | 1024 | 1 | 1e-3 | **20** | **26, 24, 40** | .7495 |
| `D_bs1` (nb 19) | 320 | 1 | 1e-3 | 40 | 94, 79, 120 | .7592 |
| `A_baseline` (nb 19) | 320 | 5 | 1e-3 | 40 | 66, 164, 58 | .8050 |

Las tres semillas a 1024 pararon a las 24-40 épocas porque llevaban `patience_es=20`.
Estaban infra-entrenadas. Y el brazo `D_bs1` del notebook 19 ya mostró que bajar el
batch cuesta **-0.046 F1 a resolución constante**.

**No existe ni un run a 1024 que se diferencie de un control a 320 solo en la
resolución.** Este notebook construye ese control.

## El diseño

`bs=1` a 1024 no fue una decisión, fue la memoria de la GPU. Pero seis imágenes de
1024 ocupan lo mismo vengan de donde vengan:

| brazo | resolución | bs | num_aug | img/paso | frames distintos |
|---|---|---|---|---|---|
| ya existe: `D_bs1` | 320 | 1 | 5 | 6 | 1 |
| **E** | 320 | 3 | 1 | 6 | **3** |
| **F** | 1024 | 3 | 1 | 6 | **3** |

**F contra E: lo único que cambia es la resolución.** E contra `D_bs1`: lo único que
cambia es de cuántos frames distintos salen las seis imágenes, que es lo que ve
BatchNorm.

`lr=1e-3` y `patience_es=40` en los dos, para no repetir el problema de arriba.

## Por qué NO se usa `num_augmented=0`

La idea original era `batch_size=6, num_augmented=0`. No sirve:

```python
# src/datasets.py:142
        if self.transform is None or self.num_augmented <= 0:
            return self._prep(image, mask, fname)
```

Con `num_augmented=0` el `return` sale **antes** de tocar la augmentación, y
`aug_profile` no lo compensa porque el corto-circuito ocurre antes. Se apagaría la
augmentación entera y volveríamos a mover dos variables. Por eso `bs=3, num_aug=1`.

## Coste, medido

```
runs/supervised_1024/seed_0     89 épocas  197.9 min -> 133.4 s/época
A_baseline (320)                66 épocas   16.8 min ->  15.3 s/época
```

1024 cuesta 8,7 veces más por época. Los 6 runs son ~11 h: ~1 h el brazo E, ~10 h el F.
Van emparejados por semilla, así que si se corta hay pares completos.


In [ ]:
# ============================================================
# SETUP - correr una vez tras cada reinicio del runtime
# No entrena nada.
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

!git clone https://github.com/sebastianquispearias/tesis-seg.git
%cd tesis-seg
!pip install -q -r requirements.txt

import torch, sys
print("Python    :", sys.version)
print("torch     :", torch.__version__)
print("CUDA      :", torch.version.cuda)
!git log --oneline -1
!nvidia-smi | grep -E "NVIDIA|Driver Version|CUDA Version"

import sys
sys.path.append("/content/tesis-seg")

import json, os, time, glob, statistics

from src.defaults import get_default_config, summarize_config
from src.augmentations import get_supervised_train_augmentation
from src.datasets import (build_supervised_datasets, build_dataloaders,
                          build_unlabeled_datasets)
from src.tee import tee_output
from src.train import run_training
from src.evaluate import evaluate_checkpoint

# --- GUARDIAN 1: el codigo clonado debe traer los perfiles nuevos ---
# El 2026-08-16 se perdieron 20 runs porque un flag se ignoro en silencio.
import inspect
from src import augmentations as _ag
from src import preprocessing as _pp
assert "get_nnunet_style_augmentation" in inspect.getsource(_ag), (
    "CODIGO VIEJO: src/augmentations.py no tiene los perfiles de nnU-Net. "
    "Borra /content/tesis-seg, vuelve a clonar y reinicia el entorno.")
assert "image_norm" in inspect.getsource(_pp.preprocess_image_and_mask), (
    "CODIGO VIEJO: preprocess_image_and_mask no acepta image_norm.")
print("OK: el codigo clonado trae aug_profile e image_norm.")

# --- GUARDIAN 2: dejar constancia del entorno ---
# Colab cambio de stack entre julio y septiembre de 2026 y albumentations 2.x
# DESCARTA EN SILENCIO argumentos que la 1.x aceptaba (var_limit, value,
# mask_value). Por eso este notebook NO compara contra runs_final_v1, que se
# entreno con albumentations 1.3.1: entrena su propio baseline en esta sesion.
import albumentations as _alb
import segmentation_models_pytorch as _smp
ENTORNO = {"python": sys.version.split()[0], "torch": torch.__version__,
           "albumentations": _alb.__version__, "smp": _smp.__version__,
           "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}
print()
print("ENTORNO DE ESTA SESION")
for k, v in ENTORNO.items():
    print("   %-16s %s" % (k, v))
print()
print("runs_final_v1 se entreno con: albumentations 1.3.1, smp 0.3.3, torch 2.2.1.")
print("Si lo de arriba no coincide, es NORMAL y por eso hay un brazo baseline aqui.")


---
### Paso 1 - Verificar ANTES de gastar 10 horas

No entrena. Comprueba que `bs=3, num_aug=1` entrega seis imágenes de tres frames
distintos, que a 1024 salen a 1024, y que un paso completo de entrenamiento a 1024
cabe en la memoria de esta GPU. Si algo falla, se sabe en dos minutos.


In [ ]:
# ============================================================
# VERIFICACION - NO ENTRENA. Tarda ~2 minutos.
# ============================================================
import numpy as np

BASE    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
ROTULOS = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"


def _cfg_brazo(cambios):
    c = get_default_config()
    c["img_root"] = BASE
    c["msk_root"] = BASE
    c["rotulos_dir"] = ROTULOS
    c["arch"] = "unetpp"
    c["backbone"] = "efficientnet-b3"
    c["n_classes"] = 1
    c["use_semi"] = False
    c["use_temp_consistency"] = False
    c["image_preproc"] = "base"
    c["mask_smoothing"] = "none"
    c["use_fixed_crop"] = False
    c["target_size"] = (320, 320)
    c["use_pad"] = True
    c["imagenet_norm"] = False
    c["batch_size"] = 5
    c["num_workers"] = 2
    c["drop_last"] = True
    c["num_augmented"] = 5
    c["lr"] = 1e-3
    c["patience_es"] = 40
    c.update(cambios)
    return c


print("PUERTA 1: el ruido de la augmentation es el pedido, no el default de la 2.x")
_gn = [t for t in get_supervised_train_augmentation(_cfg_brazo({})).transforms
       if t.__class__.__name__ == "GaussNoise"][0]
_sr = _gn.to_dict()["transform"]["std_range"]
print("   std_range =", _sr)
assert _sr[1] < 0.05, ("CODIGO VIEJO: std_range es el default (0.2, 0.44). "
                       "Borra /content/tesis-seg, vuelve a clonar y reinicia.")
print("   OK: std %.2f a %.2f niveles sobre 255" % (_sr[0] * 255, _sr[1] * 255))
print()

print("PUERTA 2: seis imagenes por paso, de TRES frames distintos, en los dos brazos")
for _nombre, _cambios in [("E  320", {"batch_size": 3, "num_augmented": 1}),
                          ("F 1024", {"batch_size": 3, "num_augmented": 1,
                                      "target_size": (1024, 1024)})]:
    _c = _cfg_brazo(_cambios)
    _tf = get_supervised_train_augmentation(_c)
    _tr, _va, _te = build_supervised_datasets(_c, train_tf=_tf)
    _ld = build_dataloaders(_c, train_ds=_tr, val_ds=_va, test_ds=_te)
    _b = next(iter(_ld["train_loader"]))
    _n = _b["image"].shape[0]
    _res = tuple(_b["image"].shape[2:])
    _frames = len(set(_b.get("name", _b.get("fname", [""] * _n))))
    assert _n == 6, f"FALLO: {_n} imagenes por paso, no 6"
    assert _res == tuple(_c["target_size"]), f"FALLO: resolucion {_res}"
    print(f"   {_nombre}: {_n} imagenes de {_frames} frames distintos, {_res}   OK")
    del _ld, _tr, _va, _te
print()

print("PUERTA 3: un paso completo a 1024 cabe en esta GPU")
import gc
from src.models import create_model

gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
_m = create_model("unetpp", "efficientnet-b3", 1).cuda().train()
_x = torch.randn(6, 3, 1024, 1024, device="cuda")
_y = torch.randint(0, 2, (6, 1, 1024, 1024), device="cuda").float()
_out = _m(_x)
_loss = torch.nn.functional.binary_cross_entropy_with_logits(_out, _y)
_loss.backward()
_pico = torch.cuda.max_memory_allocated() / 1024 ** 3
_total = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
print(f"   pico de memoria: {_pico:.1f} GB de {_total:.1f} GB")
assert _pico < _total * 0.9, "FALLO: no cabe con margen. Bajar a bs=2, num_aug=2."
print("   OK: cabe con margen")
del _m, _x, _y, _out, _loss
gc.collect(); torch.cuda.empty_cache()
print()
print("TODO VERIFICADO. Se puede entrenar.")


---
### Paso 1b - Los parámetros

Todo lo que se puede tocar está aquí y solo aquí.


In [ ]:
# ============================================================
# PARAMETROS DEL EXPERIMENTO
# ============================================================
BASE     = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
ROTULOS  = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"
OUT_ROOT = f"{BASE}/runs_resolucion"          # directorio NUEVO

SEMILLAS = [0, 1, 2]

# Los dos brazos. Mismas 6 imagenes por paso, mismos 3 frames distintos detras,
# mismo lr y misma paciencia. LO UNICO QUE CAMBIA ES target_size.
BRAZOS = [
    ("E_320_6img",  {"batch_size": 3, "num_augmented": 1}),
    ("F_1024_6img", {"batch_size": 3, "num_augmented": 1,
                     "target_size": (1024, 1024)}),
]

BRAZO_BASELINE = "E_320_6img"     # contra el que se compara la resolucion

print("brazos   :", [b[0] for b in BRAZOS])
print("semillas :", SEMILLAS)
print("salida   :", OUT_ROOT)
print("runs     : %d  (~11 h: ~1 h el brazo E, ~10 h el F)" % (len(BRAZOS) * len(SEMILLAS)))


---
### Paso 1c - El cuerpo de un run

Copiada **verbatim** del notebook 19. `cambios` es lo unico que distingue un brazo de otro.


In [ ]:
def run_arm(nombre, semilla, cambios, verbose=False):
    """Entrena un supervisado UNM aplicando `cambios` sobre la config del baseline.

    `cambios` es el unico sitio donde los brazos se diferencian. Devuelve el exp_dir.
    Si el run ya estaba completo no reentrena nada.
    """
    import gc; gc.collect()
    torch.cuda.empty_cache()

    cfg = get_default_config()
    cfg["img_root"]    = BASE
    cfg["msk_root"]    = BASE
    cfg["rotulos_dir"] = ROTULOS
    cfg["exp_dir"]     = f"{OUT_ROOT}/{nombre}/seed_{semilla}"

    cfg["arch"]      = "unetpp"
    cfg["backbone"]  = "efficientnet-b3"
    cfg["n_classes"] = 1

    cfg["seed"]                 = semilla
    cfg["use_semi"]             = False
    cfg["use_temp_consistency"] = False
    cfg["lambda_u"]             = 0.0
    cfg["lambda_t"]             = 0.0

    cfg["image_preproc"]  = "base"
    cfg["mask_smoothing"] = "none"
    cfg["use_fixed_crop"] = False
    cfg["target_size"]    = (320, 320)
    cfg["use_pad"]        = True
    cfg["imagenet_norm"]  = False

    cfg["batch_size"]     = 5
    cfg["num_workers"]    = 4
    cfg["drop_last"]      = True
    cfg["num_augmented"]  = 5
    cfg["lr"]             = 1e-3
    cfg["weight_decay"]   = 1e-4
    cfg["epochs"]         = 2000
    cfg["warmup_epochs"]  = 10
    cfg["patience_es"]    = 40
    cfg["eval_threshold"] = 0.5

    cfg["save_preds_vis"] = False
    cfg["run_ruler_eval"] = True

    # --- LO UNICO QUE DISTINGUE A ESTE BRAZO ---
    cfg.update(cambios)

    _exp = cfg["exp_dir"]
    _best    = os.path.isfile(os.path.join(_exp, "best_model.pt"))
    _metrics = os.path.isfile(os.path.join(_exp, "test_metrics.csv"))
    _summary = os.path.isfile(os.path.join(_exp, "run_summary.txt"))
    _report  = len(glob.glob(os.path.join(_exp, "*_run_report.json"))) > 0

    if _best and not _summary:
        print(f"AVISO {nombre}/seed_{semilla}: best_model.pt sin run_summary.txt.")
        print("      Run cortado a medias. Se reentrena desde cero.")

    if _best and _metrics and _summary:
        print(f"Skipping {nombre}/seed_{semilla}: run completo detectado")
        return _exp

    if verbose:
        print(summarize_config(cfg))

    # guardian en caliente: los cambios de este brazo tienen que estar en el cfg
    for k, v in cambios.items():
        assert cfg[k] == v, f"FALLO: cfg[{k}] es {cfg[k]!r}, no {v!r}"
    print(f"guardian OK: {nombre} -> {cambios}")

    train_tf = get_supervised_train_augmentation(cfg)
    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    # El pool sin etiquetar hay que construirlo aqui: build_dataloaders no lo crea,
    # solo envuelve el que se le pase. Sin esto unlabeled_loader queda en None y la
    # rama SSL no se ejecuta aunque el cfg diga use_semi=True.
    unlabeled_ds, temporal_unlab_ds = None, None
    if cfg.get("use_semi", False) or cfg.get("use_temp_consistency", False):
        unlabeled_ds, temporal_unlab_ds = build_unlabeled_datasets(cfg)
    loaders = build_dataloaders(cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds,
                                unlabeled_ds=unlabeled_ds,
                                temporal_unlab_ds=temporal_unlab_ds)

    # Guardian del EFECTO, no de la intencion. El 3 y 4 de septiembre diecinueve
    # runs se declararon Mean Teacher y entrenaron sin una imagen sin etiquetar.
    if cfg.get("use_semi", False):
        assert loaders.get("unlabeled_loader") is not None, (
            "FALLO: use_semi=True pero no se construyo el unlabeled_loader")
        assert len(loaders["unlabeled_loader"].dataset) > 0, (
            "FALLO: el pool sin etiquetar esta vacio")
        print(f"pool OK: {len(loaders['unlabeled_loader'].dataset)} frames sin "
              f"etiquetar de {cfg['unlabeled_subdir']}")

    # guardian en caliente 2: el loader de train aplana 1 vista base + num_augmented
    # vistas por frame (SegmentationDataset.__getitem__ + flatten_collate), de modo que
    # el tensor trae batch_size * (1 + num_augmented) imagenes, no batch_size.
    _bx = next(iter(loaders["train_loader"]))["image"]
    _esperado = cfg["batch_size"] * (1 + cfg["num_augmented"])
    assert _bx.shape[0] == _esperado, (
        f"FALLO: el loader entrega {_bx.shape[0]} imagenes, no {_esperado} = "
        f"batch_size {cfg['batch_size']} x (1 + num_augmented {cfg['num_augmented']}).")
    assert tuple(_bx.shape[2:]) == tuple(cfg["target_size"]), "FALLO: resolucion inesperada"
    print(f"loader OK: {_bx.shape[0]} imagenes por paso = "
          f"{cfg['batch_size']} frames x {1 + cfg['num_augmented']} vistas, {tuple(_bx.shape[2:])}")

    # Copia de todo lo que imprime el run, sin dejar de mostrarlo en pantalla.
    with tee_output(os.path.join(_exp, "stdout.log")):
        _t0 = time.time()
        if _best and _summary and (not _metrics or not _report):
            from src.models import create_model
            _m = create_model(cfg["arch"], cfg["backbone"], cfg["n_classes"])
            results = evaluate_checkpoint(cfg, _m, loaders,
                                          os.path.join(_exp, "best_model.pt"), [])
        else:
            art = run_training(cfg, loaders)
            results = evaluate_checkpoint(cfg, art["model"], loaders,
                                          art["best_path"], art["history"])
            # Tercer guardian, con el run ya terminado: la perdida no supervisada
            # tiene que haberse movido. Con lambda_u=0 sigue siendo distinta de cero,
            # asi que un cero solo puede significar que la rama no se ejecuto.
            if cfg.get("use_semi", False):
                _mu = max([(e.get("unsup_loss") or 0.0)
                           for e in (art["history"] or [])] or [0.0])
                assert _mu > 0, (
                    "FALLO: unsup_loss se quedo en cero en todas las epocas; "
                    "el run NO fue semi-supervisado")
        print(f"[{nombre}/seed_{semilla}] {(time.time()-_t0)/60:.1f} min")
        print(results)
    return _exp


print("run_arm definida.")


---
## Paso 2 - Los 6 runs (~11 h)

Van **emparejados por semilla**: E seed 0, F seed 0, E seed 1, F seed 1, y así. Si
Colab se cae de madrugada, lo que queda hecho son pares completos y comparables, no
tres runs de un brazo y ninguno del otro.

Reanudable: volver a ejecutar esta celda salta lo que ya esté terminado.


In [ ]:
# === PASO 2: los 6 runs, emparejados por semilla ===
_fallos = []
for _seed in SEMILLAS:
    for _nombre, _cambios in BRAZOS:
        print("=" * 70)
        print(f">>> {_nombre}  seed={_seed}  cambios={_cambios}")
        print("=" * 70)
        try:
            run_arm(_nombre, _seed, _cambios)
        except Exception as e:
            print(f"FALLO en {_nombre}/seed_{_seed}: {type(e).__name__}: {e}")
            _fallos.append((_nombre, _seed, repr(e)))

print()
print("terminado. fallos:", len(_fallos))
for f in _fallos:
    print("  ", f)


---
## Paso 3 - El resumen

Solo lectura. La resta que importa es **F menos E**, porque entre esos dos lo único
que cambia es la resolución. Se imprime también `D_bs1` del notebook 19, que tiene las
mismas seis imágenes por paso pero de un solo frame, para separar la resolución de la
diversidad de frames.


In [ ]:
# === PASO 3: RESUMEN (solo lectura) ===
import glob, json, os, statistics


def _f1_de(seeddir):
    ps = glob.glob(os.path.join(seeddir, "*run_report.json"))
    if not ps:
        return None
    tm = json.load(open(ps[0])).get("test_metrics", {})
    return tm.get("sample_mean_f1", tm.get("f1_mean"))


def _resumen(raiz, brazo):
    fs = [_f1_de(d) for d in sorted(glob.glob(f"{raiz}/{brazo}/seed_*"))]
    fs = [f for f in fs if f is not None]
    if not fs:
        return None, None, []
    sd = statistics.stdev(fs) if len(fs) > 1 else 0.0
    return statistics.mean(fs), sd, fs


print("%-22s %-8s %-4s %-8s %s" % ("brazo", "res", "n", "F1", "crudos"))
print("-" * 78)
_medias = {}
for _n, _c in BRAZOS:
    _res = "1024" if _c.get("target_size", (320, 320))[0] == 1024 else "320"
    _m, _s, _fs = _resumen(OUT_ROOT, _n)
    _medias[_n] = _m
    if _m is None:
        print("%-22s %-8s %-4s %s" % (_n, _res, "-", "sin resultados todavia"))
    else:
        print("%-22s %-8s %-4d %.4f+/-%.4f  %s" %
              (_n, _res, len(_fs), _m, _s, [round(f, 4) for f in _fs]))

# referencias del notebook 19, mismas 6 imagenes por paso pero de 1 solo frame
_ref = f"{BASE}/runs_nnunet_ablation"
for _n in ("D_bs1", "A_baseline", "A_baseline_fixnoise"):
    _m, _s, _fs = _resumen(_ref, _n)
    if _m is not None:
        print("%-22s %-8s %-4d %.4f+/-%.4f  %s   (nb 19)" %
              (_n, "320", len(_fs), _m, _s, [round(f, 4) for f in _fs]))

print()
_e, _f = _medias.get("E_320_6img"), _medias.get("F_1024_6img")
if _e is not None and _f is not None:
    print("LA RESTA QUE IMPORTA")
    print("   F (1024) menos E (320), a igual batch, frames, lr y paciencia: %+.4f" %
          (_f - _e))
    print()
    if abs(_f - _e) < 0.015:
        print("   Lectura: la resolucion NO explica nada. Los runs viejos a 1024")
        print("   salian peor por el batch y por la paciencia, no por la resolucion.")
    elif _f > _e:
        print("   Lectura: la resolucion SI ayuda, y nunca se habia visto porque")
        print("   los runs viejos la confundian con el batch y con la paciencia.")
    else:
        print("   Lectura: la resolucion perjudica por si sola. Ahora si se puede")
        print("   decir 'probamos 1024 y salio peor', que antes no se podia.")
else:
    print("Faltan brazos por terminar.")
